In [0]:
import pyspark.sql.functions as F

features = spark.table("nyc_taxi_analytics.fare_prediction.yellow_taxi_features")

features = features.withColumn(
    "target_fare_total",
    F.col("total_amount") - F.col("tip_amount")
)

total_rows = features.count()
print(f"Total rows in yellow_taxi_features: {total_rows}")


In [0]:
## 1 - Rows per processed month

rows_by_month = (
    features
    .groupBy("processed_year", "processing_month")
    .agg(F.count("*").alias("row_count"))
    .orderBy("processed_year", "processing_month")
)

display(rows_by_month)

In [0]:
## 2 - Target distribution (total_amount - tip_amount)

features.select("target_fare_total").summary(
    "count", "min", "25%", "50%", "75%", "90%", "99%", "max"
).show()



In [0]:
display(
    features
    .withColumn("target_bucket", F.round(F.col("target_fare_total") / 5) * 5)
    .groupBy("target_bucket")
    .agg(F.count("*").alias("trip_count"))
    .orderBy("target_bucket")
)

In [0]:
## 3 - Outliers `clean_data` doesn't filter today

speed_outliers = features.filter(
    (F.col("avg_speed_kmh") > 120) | (F.col("avg_speed_kmh") < 1)
)
print(f"Trips with avg_speed_km > 120 or < 1: {speed_outliers.count()} of {total_rows}")
display(speed_outliers.select(
    "trip_distance", "duration_minutes", "avg_speed_kmh", "target_fare_total"
).limit(20))

In [0]:
distance_outliers = features.filter(F.col("trip_distance") > 100)
print(f"Trips with trip_distance > 100 miles: {distance_outliers.count()} of {total_rows}")
display(distance_outliers.select(
    "trip_distance", "duration_minutes", "fare_amount", "tip_amount", "target_fare_total"
).limit(20))

In [0]:
high_target = features.filter(F.col("target_fare_total") > 500)
print(f"Trips with target_fare_total > $500: {high_target.count()} of {total_rows}")
display(high_target.select(
    "trip_distance", "duration_minutes", "fare_amount", "tip_amount", "target_fare_total"
    ).limit(20))

In [0]:
## 4 - Nulls in the pre-trip features

PRETRIP_FEATURES = [
    "PULocationID", "DOLocationID", "trip_distance", "passenger_count",
    "pickup_hour", "pickup_day_of_week", "pickup_month", "is_weekend",
    "is_rush_hour", "season", "pickup_manhattan", "dropoff_manhattan",
    "manhattan_trip", "is_airport_trip",
]

null_counts = features.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in PRETRIP_FEATURES
])

display(null_counts)

In [0]:
spark.table("nyc_taxi_analytics.fare_prediction.yellow_taxi_features") \
    .filter("processed_year IS NULL") \
    .groupBy(F.year("tpep_pickup_datetime").alias("y"), F.month("tpep_pickup_datetime").alias("m")) \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20)

In [0]:
spark.sql("""
    DELETE FROM nyc_taxi_analytics.fare_prediction.yellow_taxi_features
    WHERE processed_year IS NULL
""").show()